# Prompt Engineering Techniques

This notebook will guide you through 9 essential prompt engineering techniques. You'll learn by doing—implementing each technique yourself and analyzing what works.

## Objective
Explore and experiment with at least 6 prompt engineering techniques to understand their purpose, effectiveness, and practical applications.

Through hands-on experimentation, we will:
1. Compare multiple prompt engineering strategies.
2. Understand how prompt design affects model outputs.
3. Identify which techniques work best for different scenarios.
4. Apply learned techniques to real-world use cases.

## 0) Setup and Configuration

**Start here.** Run the cell below first before exploring any techniques.

In this section we will:
- Load environment variables from `.env`.
- Initialize the Azure OpenAI client.
- Verify connectivity with a minimal test call.
- Define a reusable helper function for testing prompts.

In [8]:
"""Setup cell for the notebook.

What this cell does:
1) Load credentials and configuration from .env
2) Validate required variables early
3) Create an OpenAI client for the Azure OpenAI-compatible /openai/v1 endpoint
4) Expose a reusable helper function (call_model)
5) Run a small connectivity smoke test
"""

import os
from dotenv import load_dotenv
from openai import OpenAI

# Step 1) Load local environment variables. override=True ensures notebook runs with latest .env values.
load_dotenv(override=True)

AZURE_API_KEY = os.getenv("AZURE_API_KEY", "").strip()
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT", "").strip()
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "").strip()

# Step 2) Fail fast if required settings are missing.
missing = [
    name
    for name, value in [
        ("AZURE_API_KEY", AZURE_API_KEY),
        ("AZURE_ENDPOINT", AZURE_ENDPOINT),
        ("DEPLOYMENT_NAME", DEPLOYMENT_NAME),
    ]
    if not value
]

if missing:
    raise ValueError(
        "Missing required environment variables: " + ", ".join(missing) +
        ". Fill them in your .env file and rerun this cell."
    )

print("Loaded environment variables successfully.")
print(f"Endpoint: {AZURE_ENDPOINT[:50]}...")
print(f"Model deployment: {DEPLOYMENT_NAME}")

# Step 3) Initialize the model client.
# The endpoint already includes /openai/v1, so the OpenAI client is the correct fit here.
client = OpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
)
print("Client initialized: OpenAI (Azure OpenAI-compatible endpoint)")


def call_model(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.7,
    max_tokens: int = 500,
    print_prompts: bool = True,
) -> str:
    """Send a chat completion request and return assistant text content."""
    if print_prompts:
        print("\n" + "=" * 60)
        print("SYSTEM PROMPT:")
        print(system_prompt)
        print("\nUSER PROMPT:")
        print(user_prompt)
        print("=" * 60 + "\n")

    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# Step 4) Smoke test to confirm connectivity before the rest of the notebook.
try:
    test_output = call_model(
        system_prompt="You are a concise assistant.",
        user_prompt="Reply with exactly: Connection OK",
        temperature=0.0,
        max_tokens=20,
        print_prompts=False,
    )
    print("✓ Connectivity test passed.")
    print(f"Test response: {test_output}")
except Exception as exc:
    print("✗ Connectivity test failed.")
    print(f"Error type: {type(exc).__name__}")
    print(f"Error detail: {exc}")

Loaded environment variables successfully.
Endpoint: https://02-prompt-engineering-resource.openai.azur...
Model deployment: gpt-4o
Client initialized: OpenAI (Azure OpenAI-compatible endpoint)
✓ Connectivity test passed.
Test response: Connection OK


---

## 1) Prompt Engineering Techniques

Below, we will test 9 prompt engineering techniques. For each technique, you will:
- **Explain it**: Describe the technique in your own words.
- **Implement it**: Write prompts and test with the model.
- **Analyze it**: Evaluate what worked and why.

### Technique 1: Person / Role

**What is it?**
This technique involves giving the model a specific role or persona before asking your question.

#### Explanation
Assigning a role narrows the model's focus to a domain, resulting in more accurate, tailored responses.

In [9]:
"""Technique 1 Implementation.

Your task:
1) Define a system prompt that assigns a specific role to the model.
2) Define a user prompt with what you actually want to ask.
3) Uncomment and run to compare with and without the role.
"""

system_prompt_1 = """
You're a travel expert assistant with deep knowledge of global destinations, local cultures, and travel logistics. You provide personalized recommendations based on user preferences and interests.
"""

user_prompt_1 = """
What are 3 locations I shouldn't miss if I visit Switzerland for the first time?
"""

response_1 = call_model(system_prompt_1, user_prompt_1)
print("MODEL OUTPUT:")
print(response_1)


SYSTEM PROMPT:

You're a travel expert assistant with deep knowledge of global destinations, local cultures, and travel logistics. You provide personalized recommendations based on user preferences and interests.


USER PROMPT:

What are 3 locations I shouldn't miss if I visit Switzerland for the first time?


MODEL OUTPUT:
Switzerland is a stunning destination with breathtaking scenery, charming towns, and rich culture. For a first-time visit, here are three must-see locations that should be on your itinerary:

---

### 1. **Zermatt and the Matterhorn**
   - **Why Visit?** The iconic Matterhorn is one of Switzerland's most famous landmarks, and Zermatt is a charming car-free alpine village nestled at its base. It's a paradise for outdoor enthusiasts, offering world-class skiing, hiking, and panoramic views. 
   - **Top Experiences:**
     - Take the **Gornergrat Railway** for breathtaking views of the Matterhorn and surrounding peaks.
     - Hike or ski on the many trails and slopes 

#### Critical Analysis
**Reflect on what you observed:**
- Did the role actually influence the response? How?
- What would happen if you removed the role from the system prompt?
- When would you use this technique in your own work?

Giving the model a specific role in the system prompt produced more accurate and relevant responses for questions within a defined domain.

If the role is removed, the response becomes more generic and usually less detailed or precise.

I would use this technique whenever I need focused, domain-specific answers rather than broad general information.

---

### Technique 2: Explicitness

**What is it?**
Being very specific and clear about what you want, including format, constraints, and expectations.

#### Explanation
Explicit prompts eliminate ambiguity through specific requirements, producing structured, predictable outputs.

In [12]:
"""Technique 2 Implementation.

Your task:
1) Write a vague prompt and an explicit prompt for the same task.
2) Test both and compare the outputs.
3) Note which one gives better results.
"""

system_prompt_2 = """
You are an AI governance assistant.
"""

# Vague prompt
user_prompt_2_vague = """
Give me good AI practices.
"""

# Explicit prompt
user_prompt_2_explicit = """
Provide 8 good AI practices for teams building internal business assistants.
Requirements:
- Group practices under these headings: Data, Model, Safety, and Operations.
- Include exactly 2 bullet points per heading.
- Each bullet must be one sentence (max 18 words).
- Use clear, actionable language that starts with a verb.
- Include one concrete example under Safety.
- Avoid generic advice and avoid repeating ideas.
"""

print("=== VAGUE PROMPT ===")
print(user_prompt_2_vague.strip())
response_2_vague = call_model(system_prompt_2, user_prompt_2_vague)
print("\nVAGUE OUTPUT:\n")
print(response_2_vague)

print("\n" + "=" * 70 + "\n")

print("=== EXPLICIT PROMPT ===")
print(user_prompt_2_explicit.strip())
response_2_explicit = call_model(system_prompt_2, user_prompt_2_explicit)
print("\nEXPLICIT OUTPUT:\n")
print(response_2_explicit)

=== VAGUE PROMPT ===
Give me good AI practices.

SYSTEM PROMPT:

You are an AI governance assistant.


USER PROMPT:

Give me good AI practices.



VAGUE OUTPUT:

Certainly! Implementing good practices in the development, deployment, and governance of AI systems is essential to ensure that they are ethical, safe, and beneficial for society. Below are some key principles and practices for AI:

---

### **1. Ethical AI Development**
- **Fairness and Non-discrimination**: Ensure AI systems do not perpetuate or amplify biases. Train models on diverse and representative datasets, and regularly audit for bias.
- **Transparency**: Provide clear and understandable documentation for how the AI system works, including its decision-making processes, limitations, and risks.
- **Accountability**: Define clear roles and responsibilities for those involved in the development and deployment of AI systems. Establish mechanisms for addressing harm or unintended consequences.
- **Respect for Privacy**: Pr

#### Critical Analysis
**Reflect on what you observed:**
- How different were the vague and explicit versions?
- Which version was more useful for your goal?
- When would you prioritize being explicit vs. being brief?

The explicit prompt produced a much clearer and more structured response than the vague one.

The vague version was broad and generic, while the explicit version followed the exact format and constraints requested.

I would prioritize explicit prompts when I need consistent, usable outputs, and keep brief prompts for open-ended brainstorming.

---

### Technique 3: Few Shot

**What is it?**
Show the model 2-3 examples of what you want, then ask it to apply the same pattern to new cases.

#### Explanation
Showing examples of the desired pattern and format before the task teaches the model what output structure you expect.

In [13]:
"""Technique 3 Implementation.

Your task:
1) Provide 2-3 examples of input/output pairs.
2) Ask the model to apply the same pattern to new inputs.
3) Check if the outputs match your expected format.
"""

system_prompt_3 = """
You are an automotive advisor focused on German cars.
Always return output in this format:
- Model:
- Segment:
- Why it fits (max 20 words):
"""

user_prompt_3 = """
Use the pattern from the examples and apply it to the new cases.

EXAMPLE 1
Input: Budget 42000 EUR, mostly highway commuting, wants efficiency.
Output:
- Model: BMW 320d
- Segment: Sedan
- Why it fits (max 20 words): Efficient diesel, comfortable at high speeds, strong long-distance reliability.

EXAMPLE 2
Input: Budget 55000 EUR, small family, wants safety and cargo space.
Output:
- Model: Audi Q5
- Segment: SUV
- Why it fits (max 20 words): Spacious, high safety ratings, premium comfort, practical for daily family use.

EXAMPLE 3
Input: Budget 70000 EUR, wants a sporty weekend car.
Output:
- Model: Porsche 718 Cayman
- Segment: Sports Coupe
- Why it fits (max 20 words): Sharp handling, strong performance, engaging drive, premium build quality.

Now apply the same pattern to:
1) Budget 50000 EUR, urban driving, wants compact premium comfort.
2) Budget 65000 EUR, long road trips, wants technology and comfort.
"""

response_3 = call_model(system_prompt_3, user_prompt_3)
print("MODEL OUTPUT:")
print(response_3)


SYSTEM PROMPT:

You are an automotive advisor focused on German cars.
Always return output in this format:
- Model:
- Segment:
- Why it fits (max 20 words):


USER PROMPT:

Use the pattern from the examples and apply it to the new cases.

EXAMPLE 1
Input: Budget 42000 EUR, mostly highway commuting, wants efficiency.
Output:
- Model: BMW 320d
- Segment: Sedan
- Why it fits (max 20 words): Efficient diesel, comfortable at high speeds, strong long-distance reliability.

EXAMPLE 2
Input: Budget 55000 EUR, small family, wants safety and cargo space.
Output:
- Model: Audi Q5
- Segment: SUV
- Why it fits (max 20 words): Spacious, high safety ratings, premium comfort, practical for daily family use.

EXAMPLE 3
Input: Budget 70000 EUR, wants a sporty weekend car.
Output:
- Model: Porsche 718 Cayman
- Segment: Sports Coupe
- Why it fits (max 20 words): Sharp handling, strong performance, engaging drive, premium build quality.

Now apply the same pattern to:
1) Budget 50000 EUR, urban driving, w

#### Critical Analysis
**Reflect on what you observed:**
- Did the few-shot examples help the model understand the pattern?
- Were the outputs consistent with your examples?
- When is few-shot better than just giving instructions?

The examples clearly helped the model understand both the content and the output format.

The responses were much more consistent with the expected structure than with instructions alone.

I would use few-shot when format consistency matters, like classification, extraction, or recommendation templates.

---

### Technique 4: Chain of Thought

**What is it?**
Ask the model to show its reasoning step-by-step before giving a final answer.

#### Explanation
Asking the model to show step-by-step reasoning before the final answer makes complex problems easier to verify and trust.

In [14]:
"""Technique 4 Implementation.

Your task:
1) Ask a question that requires reasoning or calculation.
2) Request step-by-step explanation before the final answer.
3) Compare with and without the step-by-step instruction.
"""

system_prompt_4 = """
You are a hiking trip planning assistant.
"""

problem_4 = """
I plan a mountain hike of 14 km with 900 m elevation gain.
Assume flat pace is 5 km/h, elevation penalty is 1 extra hour per 600 m ascent,
and I will take two breaks of 15 minutes each.
Estimate total hiking time.
"""

user_prompt_4_brief = f"""
{problem_4}
Give only the final estimated time.
"""

user_prompt_4_cot = f"""
{problem_4}
Show your work step-by-step before giving the final answer.
"""

print("=== WITHOUT STEP-BY-STEP ===")
response_4_brief = call_model(system_prompt_4, user_prompt_4_brief)
print(response_4_brief)

print("\n" + "=" * 70 + "\n")

print("=== WITH STEP-BY-STEP ===")
response_4_cot = call_model(system_prompt_4, user_prompt_4_cot)
print(response_4_cot)

=== WITHOUT STEP-BY-STEP ===

SYSTEM PROMPT:

You are a hiking trip planning assistant.


USER PROMPT:


I plan a mountain hike of 14 km with 900 m elevation gain.
Assume flat pace is 5 km/h, elevation penalty is 1 extra hour per 600 m ascent,
and I will take two breaks of 15 minutes each.
Estimate total hiking time.

Give only the final estimated time.


4 hours 50 minutes


=== WITH STEP-BY-STEP ===

SYSTEM PROMPT:

You are a hiking trip planning assistant.


USER PROMPT:


I plan a mountain hike of 14 km with 900 m elevation gain.
Assume flat pace is 5 km/h, elevation penalty is 1 extra hour per 600 m ascent,
and I will take two breaks of 15 minutes each.
Estimate total hiking time.

Show your work step-by-step before giving the final answer.


To estimate your total hiking time, we will break the calculation into steps:

---

### Step 1: Calculate the time for the flat distance
Your total hike distance is 14 km. Assuming a flat pace of 5 km/h, the time required to cover this distan

#### Critical Analysis
**Reflect on what you observed:**
- Did step-by-step reasoning improve the answer quality?
- Could you verify the logic more easily?
- For what types of problems is this most useful?

The step-by-step version was easier to trust because I could follow each part of the calculation.

Without the reasoning, I only got a number and could not verify where it came from.

I would use this technique for math, planning, or any task where intermediate logic matters.

---

### Technique 5: Constraint

**What is it?**
Set specific rules, limits, or requirements that the response must follow.

#### Explanation
Defining strict boundaries for response format, length, and tone controls output predictability and quality.

In [15]:
"""Technique 5 Implementation.

Your task:
1) Set 2-3 specific constraints for your response.
2) Test if the model follows all of them.
3) Identify which constraints are easiest/hardest to follow.
"""

system_prompt_5 = """
You are a fitness and preventive healthcare coach.
"""

user_prompt_5 = """
Create a 1-day beginner wellness plan for an office worker.
Constraints:
- Exactly 5 bullet points.
- Each bullet must start with an action verb.
- Each bullet must be 12 words or fewer.
- Include one hydration action and one mobility action.
- Keep the tone practical and realistic.
"""

response_5 = call_model(system_prompt_5, user_prompt_5)
print("MODEL OUTPUT:")
print(response_5)


SYSTEM PROMPT:

You are a fitness and preventive healthcare coach.


USER PROMPT:

Create a 1-day beginner wellness plan for an office worker.
Constraints:
- Exactly 5 bullet points.
- Each bullet must start with an action verb.
- Each bullet must be 12 words or fewer.
- Include one hydration action and one mobility action.
- Keep the tone practical and realistic.


MODEL OUTPUT:
- Start your day with a 5-minute stretch to release morning tension.  
- Drink a glass of water before starting work to stay hydrated.  
- Take a 5-minute walk every hour to reduce stiffness and boost focus.  
- Eat a balanced lunch with protein, veggies, and whole grains for energy.  
- Wind down with 10 minutes of deep breathing or light yoga before bed.  


#### Critical Analysis
**Reflect on what you observed:**
- Did the model follow all your constraints?
- Which constraints were respected more consistently?
- Did constraints improve or limit the quality of responses?

Most constraints were followed, especially format and action-oriented wording.

The harder part for the model was staying within strict word limits while still being specific.

Overall, constraints improved clarity and usefulness, although they can reduce creativity if too strict.

---

### Technique 6: Prompt Splitting / Decomposition

**What is it?**
Break a complex problem into smaller, simpler prompts that build on each other.

#### Explanation
Breaking complex problems into smaller, sequential steps provides better control by letting you validate each intermediate result.

In [16]:
"""Technique 6 Implementation.

Your task:
1) Take a complex problem and split it into 2-3 smaller prompts.
2) Call the model multiple times, building on previous responses.
3) Compare the final result with asking the full question at once.
"""

system_prompt_6 = """
You are a travel planning assistant specialized in Europe trips.
"""

user_prompt_6_step1 = """
Suggest the best 3 cities in Germany for a first-time traveler interested in history,
food, and efficient train connections. Give one reason per city.
"""

response_6_step1 = call_model(system_prompt_6, user_prompt_6_step1)
print("STEP 1 OUTPUT:")
print(response_6_step1)

user_prompt_6_step2 = f"""
Based on this shortlist:
{response_6_step1}

Build a realistic 6-day itinerary with:
- Day-by-day city allocation
- One main activity per day
- Estimated train transfer notes
"""

response_6_step2 = call_model(system_prompt_6, user_prompt_6_step2)
print("\nSTEP 2 OUTPUT:")
print(response_6_step2)

user_prompt_6_single = """
Create a full 6-day Germany itinerary for a first-time traveler interested in history,
food, and easy train transfers. Include city choices and day-by-day plan.
"""

response_6_single = call_model(system_prompt_6, user_prompt_6_single)
print("\n" + "=" * 70 + "\n")
print("SINGLE-PROMPT OUTPUT:")
print(response_6_single)


SYSTEM PROMPT:

You are a travel planning assistant specialized in Europe trips.


USER PROMPT:

Suggest the best 3 cities in Germany for a first-time traveler interested in history,
food, and efficient train connections. Give one reason per city.


STEP 1 OUTPUT:
1. **Berlin**: A hub of historical significance with iconic landmarks like the Brandenburg Gate, Berlin Wall, and Museum Island, offering a deep dive into Germany’s past.  
   
2. **Munich**: Famous for its Bavarian cuisine, beer gardens, and traditional markets, it’s a must-visit for food lovers and cultural enthusiasts alike.  

3. **Nuremberg**: Known for its well-preserved medieval architecture and historical sites like the Nuremberg Trials Memorial, it is easily accessible by train and perfect for history buffs.

SYSTEM PROMPT:

You are a travel planning assistant specialized in Europe trips.


USER PROMPT:

Based on this shortlist:
1. **Berlin**: A hub of historical significance with iconic landmarks like the Brandenbu

#### Critical Analysis
**Reflect on what you observed:**
- Was decomposition more effective than asking everything at once?
- Did each step improve the final result?
- What types of problems benefit from decomposition?

Decomposition gave me better control over the final result than a single large prompt.

Each step added useful structure, and I could correct direction before generating the final itinerary.

This works best for workflows with multiple stages, like planning, research synthesis, or structured content generation.

---

### Technique 7: Meta Prompting

**What is it?**
Ask the model to reflect on, evaluate, or critique its own responses before giving a final answer.

#### Explanation
Asking the model to review and improve its own output catches vague language and weak structure through self-reflection.

In [17]:
"""Technique 7 Implementation.

Your task:
1) Ask the model to solve a problem.
2) Then ask it to review and critique its own answer.
3) Compare the original and revised responses.
"""

system_prompt_7 = """
You are a health and fitness planning assistant.
Avoid medical diagnosis and provide general wellness guidance only.
"""

user_prompt_7_initial = """
Create a weekly routine for a beginner who wants better fitness and stress control.
Include sleep, training, and recovery habits.
"""

response_7_initial = call_model(system_prompt_7, user_prompt_7_initial)
print("INITIAL RESPONSE:")
print(response_7_initial)

user_prompt_7_critique = f"""
Review your previous response:
{response_7_initial}

Identify weak points, missing details, or unclear advice.
Then provide an improved version with better structure and practicality.
"""

response_7_critique = call_model(system_prompt_7, user_prompt_7_critique)
print("\nREVISED RESPONSE:")
print(response_7_critique)


SYSTEM PROMPT:

You are a health and fitness planning assistant.
Avoid medical diagnosis and provide general wellness guidance only.


USER PROMPT:

Create a weekly routine for a beginner who wants better fitness and stress control.
Include sleep, training, and recovery habits.


INITIAL RESPONSE:
Here’s a simple, balanced weekly routine for a beginner who wants to improve fitness and manage stress. It incorporates physical activity, sleep, recovery, and mindfulness practices for stress control.

---

### **General Guidelines**
- **Sleep:** Aim for 7–9 hours of uninterrupted sleep each night. Go to bed and wake up at the same time daily.
- **Hydration:** Drink plenty of water throughout the day (aim for 2–3 liters, depending on your needs).
- **Nutrition:** Opt for balanced meals with lean proteins, whole grains, healthy fats, and plenty of fruits and vegetables.
- **Recovery:** Include rest days and light activities to support muscle recovery and mental relaxation.
- **Progression:**

#### Critical Analysis
**Reflect on what you observed:**
- Did self-reflection improve the model's responses?
- What kinds of errors did the model catch about itself?
- Is meta-prompting useful for all types of tasks?

The self-review step improved clarity and made the plan more practical.

The model mostly fixed missing specifics and replaced vague statements with clearer actions.

Meta-prompting is useful for quality improvement, but it is less necessary for simple one-step tasks.

---

### Technique 8: Directional Stimulus

**What is it?**
Guide the model toward a specific tone, style, or perspective by using directional hints or starting phrases.

#### Explanation
Using style cues, starting phrases, and perspective hints shapes the model's voice and tone without changing the core task.

In [18]:
"""Technique 8 Implementation.

Your task:
1) Give the model a directive about tone or style.
2) Provide starting hints that guide the response direction.
3) Test how well directional stimuli influence the output.
"""

system_prompt_8 = """
You are a travel content writer.
"""

user_prompt_8 = """
Write a short post about a sunrise hiking route in the Swiss Alps.
Directional hints:
- Tone: energetic, encouraging, and practical.
- Perspective: speak like an experienced hiking buddy.
- Begin with: "If you want one hike that feels unreal, start here."
- Focus on: preparation, safety, and why the route is memorable.
- Keep it under 140 words.
"""

response_8 = call_model(system_prompt_8, user_prompt_8)
print("MODEL OUTPUT:")
print(response_8)


SYSTEM PROMPT:

You are a travel content writer.


USER PROMPT:

Write a short post about a sunrise hiking route in the Swiss Alps.
Directional hints:
- Tone: energetic, encouraging, and practical.
- Perspective: speak like an experienced hiking buddy.
- Begin with: "If you want one hike that feels unreal, start here."
- Focus on: preparation, safety, and why the route is memorable.
- Keep it under 140 words.


MODEL OUTPUT:
If you want one hike that feels unreal, start here: the trail to Schynige Platte in the Swiss Alps at sunrise. Trust me, the views will blow your mind—glowing peaks, misty valleys, and lakes shimmering like jewels. Start early (think headlamp early!) to catch the alpine world waking up. Layer up—it’s chilly before the sun hits, and sturdy boots are a must on the rocky path. Pack snacks, water, and that extra battery for your camera—you’ll need it! The route is moderately challenging, but the payoff? Unreal panoramic views of Eiger, Mönch, and Jungfrau. Don’t rush.

#### Critical Analysis
**Reflect on what you observed:**
- Did directional hints influence the tone and style?
- How specific do hints need to be?
- Which types of hints are most effective?

The hints clearly changed the tone and made the response sound more intentional.

Specific starting phrases and perspective cues had the strongest effect.

General hints helped, but precise direction produced the most reliable style control.

---

### Technique 9: Self-Consistency

**What is it?**
Generate multiple independent answers for the same task and pick the most frequent or most consistent one.

#### Explanation
Running the same prompt multiple times and selecting the most frequent result reduces randomness and produces more stable answers.

In [20]:
"""Technique 9 Implementation.

Your task:
1) Ask the same question multiple times with controlled randomness.
2) Collect all candidate answers.
3) Choose the most frequent answer (majority vote).
"""

from collections import Counter

system_prompt_9 = """
You are a concise travel assistant specialized in German city recommendations.
"""

user_prompt_9 = """
A first-time visitor has 3 days in Germany and wants history, walkability, and museums.
Choose ONE best city: Berlin, Munich, Hamburg, or Cologne.
Return exactly one city name with no extra words.
"""

candidates = []
for i in range(7):
    answer = call_model(
        system_prompt_9,
        user_prompt_9,
        temperature=0.8,
        max_tokens=8,
        print_prompts=False,
    )
    city = answer.strip().splitlines()[0].strip()
    candidates.append(city)
    print(f"Run {i + 1}: {city}")

vote_counts = Counter(candidates)
final_city, votes = vote_counts.most_common(1)[0]

print("\n" + "=" * 70)
print("VOTE SUMMARY:")
for city, count in vote_counts.items():
    print(f"- {city}: {count}")

print(f"\nFINAL ANSWER (self-consistency): {final_city} ({votes}/7 votes)")

Run 1: Berlin
Run 2: Berlin
Run 3: Berlin
Run 4: Berlin
Run 5: Berlin
Run 6: Berlin
Run 7: Berlin

VOTE SUMMARY:
- Berlin: 7

FINAL ANSWER (self-consistency): Berlin (7/7 votes)


#### Critical Analysis
**Reflect on what you observed:**
- Did running multiple attempts improve result stability?
- Was the majority vote clearer than a single run?
- When would you use self-consistency in practice?

The multi-run approach gave a more stable final answer than trusting one response.

Majority voting reduced the effect of random variation between runs.

I would use self-consistency for high-impact decisions or reasoning tasks where reliability matters more than speed.

---

## 2) Real Use Cases

Here I apply the strongest techniques from Part 1 to two practical problems: product review analysis and customer support triage.

### Use Case 1: Product Review Analysis

**The Problem:**
You have customer product reviews and need to quickly identify sentiment (positive, negative, mixed) and extract specific features mentioned (build quality, design, value, performance, etc.). A single model call won't give you consistent, structured output.

**Techniques Used:**
- **Role/Person (1)** - Assign the model a clear analytical role
- **Explicitness (2)** - Specify exact output format and required fields
- **Constraint (5)** - Enforce exact categories and structure
- **Few-Shot (3)** - Show 2 examples of correct analysis

In [21]:
"""Real Use Case 1: Product Review Analysis

Combined Techniques:
1) Role assignment (Technique 1) - make the model a product analyst
2) Explicitness (Technique 2) - define exact output format
3) Constraint (Technique 5) - enforce specific categories
4) Few-Shot (Technique 3) - show 2 examples before asking for analysis
"""

system_prompt_uc1 = """
You are a product review analyst skilled at extracting structured insights from unstructured customer feedback.
Your role is to identify sentiment and key product features mentioned in reviews.
Be precise, objective, and consistent in your analysis.
"""

user_prompt_uc1 = """
Analyze each review and return structured output. Follow these rules exactly:

OUTPUT FORMAT:
Sentiment: [positive | negative | mixed]
Features Mentioned: [comma-separated list]
Key Quote: [one representative sentence from the review]
Recommendation: [keep | improve | discontinue]

EXAMPLES:

REVIEW EXAMPLE 1:
"Love this laptop! Super fast, sleek design, long battery life. Keyboard is noisy but overall fantastic for the price."

ANALYSIS EXAMPLE 1:
Sentiment: mixed
Features Mentioned: speed, design, battery life, keyboard, price
Key Quote: "Super fast, sleek design, long battery life."
Recommendation: keep

REVIEW EXAMPLE 2:
"Waste of money. Screen broke after 2 weeks. Customer service was unhelpful. Would not recommend to anyone."

ANALYSIS EXAMPLE 2:
Sentiment: negative
Features Mentioned: durability, build quality, customer service
Key Quote: "Screen broke after 2 weeks."
Recommendation: improve

---

Now analyze these reviews:

REVIEW 1:
"The camera quality is exceptional, especially in low light. Photos are crisp and detailed. My only complaint is the phone gets warm during video recording. Otherwise solid product."

REVIEW 2:
"Terrible phone. Screen is dim, camera is blurry, and the interface lags constantly. Returned it after one week. Save your money."
"""

print("=" * 70)
print("REAL USE CASE 1: Product Review Analysis")
print("=" * 70)
print("\nAnalyzing two product reviews using combined techniques...\n")

response_uc1 = call_model(
    system_prompt_uc1,
    user_prompt_uc1,
    temperature=0.3,
    max_tokens=400,
)

print("ANALYSIS OUTPUT:")
print(response_uc1)

REAL USE CASE 1: Product Review Analysis

Analyzing two product reviews using combined techniques...


SYSTEM PROMPT:

You are a product review analyst skilled at extracting structured insights from unstructured customer feedback.
Your role is to identify sentiment and key product features mentioned in reviews.
Be precise, objective, and consistent in your analysis.


USER PROMPT:

Analyze each review and return structured output. Follow these rules exactly:

OUTPUT FORMAT:
Sentiment: [positive | negative | mixed]
Features Mentioned: [comma-separated list]
Key Quote: [one representative sentence from the review]
Recommendation: [keep | improve | discontinue]

EXAMPLES:

REVIEW EXAMPLE 1:
"Love this laptop! Super fast, sleek design, long battery life. Keyboard is noisy but overall fantastic for the price."

ANALYSIS EXAMPLE 1:
Sentiment: mixed
Features Mentioned: speed, design, battery life, keyboard, price
Key Quote: "Super fast, sleek design, long battery life."
Recommendation: keep



#### Critical Analysis

**What worked:**
Decomposing the ticket into steps forced the model to think through the problem methodically. The second step produced well-structured resolutions because the model had already identified issues clearly in Step 1.

**Trade-offs:**
Multi-step decomposition required 2 API calls instead of 1, but the quality improvement was significant. For high-volume support systems, this method scales well because the structured output can be routed automatically.

**When to use this pattern:**
This works well for any complex analysis that benefits from intermediate validation: technical troubleshooting, financial audits, content moderation, or task-specific workflows where intermediate steps are worth reviewing.

### Use Case 2: Customer Support Ticket Resolution

**Techniques Used:**
- **Decomposition (6)** - Split ticket analysis into logical steps
- **Chain of Thought (4)** - Show reasoning before recommending resolution
- **Role/Person (1)** - Assign expert support agent role
- **Constraint (5)** - Enforce max 3 issues per ticket, specific category list

In [22]:
"""Real Use Case 2: Customer Support Ticket Resolution

Combined Techniques:
1) Decomposition (Technique 6) - break ticket into steps
2) Chain of Thought (Technique 4) - show reasoning
3) Role (Technique 1) - expert support agent
4) Constraint (Technique 5) - enforce categories and structure
"""

system_prompt_uc2 = """
You are an expert customer support agent with 10 years of experience.
Your job is to analyze support tickets, categorize issues, and recommend resolution steps.
Always reason through the ticket clearly before making recommendations.
"""

user_prompt_uc2_step1 = """
Read this support ticket and identify the core issues:

TICKET:
Subject: App crashes on startup and won't save my data
Body: Hi, I updated to the latest version yesterday and now the app crashes every time I try to open it. I had important data in there that I haven't backed up yet. I'm on Android 12 with a Pixel 6. This is urgent!

STEP 1: List all issues mentioned in the ticket (be specific).
"""

response_uc2_step1 = call_model(
    system_prompt_uc2,
    user_prompt_uc2_step1,
    temperature=0.5,
    max_tokens=300,
)

print("=" * 70)
print("REAL USE CASE 2: Support Ticket Resolution")
print("=" * 70)
print("\nSTEP 1: Issue Identification")
print(response_uc2_step1)

user_prompt_uc2_step2 = f"""
Based on the issues you identified:
{response_uc2_step1}

STEP 2: Now categorize each issue and recommend resolution steps.

Use this exact format:

ISSUE #1:
- Category: [bug | feature-request | account | payment | other]
- Severity: [low | medium | high | critical]
- Resolution Steps:
  1) [specific first step]
  2) [specific second step]
  3) [specific third step if needed]
- Escalate to: [no | engineering | data-recovery | senior-support]

Repeat for each issue. Keep resolution steps practical and actionable.
"""

response_uc2_step2 = call_model(
    system_prompt_uc2,
    user_prompt_uc2_step2,
    temperature=0.4,
    max_tokens=500,
)

print("\nSTEP 2: Issue Categorization and Resolution")
print(response_uc2_step2)


SYSTEM PROMPT:

You are an expert customer support agent with 10 years of experience.
Your job is to analyze support tickets, categorize issues, and recommend resolution steps.
Always reason through the ticket clearly before making recommendations.


USER PROMPT:

Read this support ticket and identify the core issues:

TICKET:
Subject: App crashes on startup and won't save my data
Body: Hi, I updated to the latest version yesterday and now the app crashes every time I try to open it. I had important data in there that I haven't backed up yet. I'm on Android 12 with a Pixel 6. This is urgent!

STEP 1: List all issues mentioned in the ticket (be specific).


REAL USE CASE 2: Support Ticket Resolution

STEP 1: Issue Identification
### Issues Mentioned in the Ticket:
1. **App crashes on startup:** The user is unable to open the app after updating to the latest version.
2. **Loss of access to important data:** The user has unbacked data stored in the app that they cannot retrieve due to th

#### Critical Analysis
**Reflect on what you observed:**
- Did running multiple attempts improve result stability?
- Was the majority vote clearer than a single run?
- When would you use self-consistency in practice?

The multi-run approach gave a more stable final answer than trusting one response.

Majority voting reduced the effect of random variation between runs.

I would use self-consistency for high-impact decisions or reasoning tasks where reliability matters more than speed.

---

## 3) Personal Conclusions

Reflect on your experimentation.

### Most Useful Techniques

The most effective techniques were **Explicitness (2)**, **Few-Shot (3)**, and **Chain of Thought (4)**. Explicitness made output predictable, Few-Shot taught patterns for structured tasks, and Chain of Thought enabled verification of logic. These three form a foundation for most use cases.

### Techniques for Different Scenarios

- **Creative Writing**: Directional Stimulus (8) + Constraint (5) + Meta Prompting (7)
- **Technical Documentation**: Role (1) + Decomposition (6) + Few-Shot (3)
- **Data Extraction**: Few-Shot (3) + Explicitness (2) + Constraint (5)
- **Code Review**: Role (1) + Chain of Thought (4) + Decomposition (6)
- **Analysis & Reports**: Decomposition (6) + Chain of Thought (4) + Meta Prompting (7)

### Key Learnings

- **Specificity wins**: Explicit, detailed prompts consistently outperform vague ones.
- **Combine 2-3 techniques**: Single techniques underperform; Few-Shot + Explicitness + Constraints produces better results than any alone.
- **Trade-offs**: Higher structure = less creativity. Multi-step approaches = better quality but higher cost. Pick based on your priorities.
- **Single vs. Combined**: Use one technique for exploration; combine for production work where consistency matters.